# Caso J · 02 Inferencia YOLO (mock por defecto)

> _Tutorial · Caso de uso: **J — Tráfico + YOLO** · Capa Medallion: **bronce → plata** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Implementar la función `count_vehicles(image)` con un mock determinista; documentar cómo conectar `ultralytics` real.


## 2. Qué se aprende

- Firma estable: bytes → dict.
- Confidence threshold y NMS.
- Cómo manejar imagen corrupta.


## 3. Contexto del caso de uso

El equipo J entrega counts en InfluxDB.


## 4. Relación con CENTINELA+

El conteo es analog_gauge.


## 5. Relación con Medallion

Bronce → plata: conteo en plata.


## 6. Datos de entrada

JPEG mock.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

`variable ∈ {vehicle_count, congestion_level, detection_confidence}`.


## 9. Carga de datos o mock

Definimos un mock determinista de YOLO.


In [2]:
def count_vehicles_mock(image_bytes: bytes, *, threshold: float = 0.4) -> dict:
    rng = np.random.default_rng(int.from_bytes(image_bytes[:4], "big") % 10000)
    n = int(rng.integers(0, 60))
    conf = float(np.clip(0.7 + rng.normal(0, 0.05), 0.4, 0.99))
    cong = int(np.clip(np.digitize([n], [10, 30, 60])[0], 0, 3))
    return {"vehicle_count": n, "detection_confidence": conf, "congestion_level": cong}

print(count_vehicles_mock(b"hello-world-bytes" * 4))


{'vehicle_count': 38, 'detection_confidence': 0.7530791527810243, 'congestion_level': 2}


## 10. Exploración paso a paso

Probamos con varias imágenes.


In [3]:
import io
from PIL import Image

results = []
for seed in range(5):
    rng = np.random.default_rng(seed)
    img = Image.new("RGB", (32, 32), (int(rng.integers(0, 255)), 0, 0))
    buf = io.BytesIO(); img.save(buf, format="JPEG")
    results.append(count_vehicles_mock(buf.getvalue()))
pd.DataFrame(results)


,vehicle_count,detection_confidence,congestion_level
0,20,0.6512,1
1,20,0.6512,1
2,20,0.6512,1
3,20,0.6512,1
4,20,0.6512,1


## 11. Transformación bronce → plata

Notebook 03.


## 12. Construcción de capa oro

Adapter al modelo real.


In [4]:
def count_vehicles_real(image_bytes: bytes, *, threshold: float = 0.4):
    """Adapter: requiere ultralytics + modelo. No se ejecuta sin instalación."""
    from ultralytics import YOLO  # type: ignore[import-not-found]
    model = YOLO("yolov8n.pt")
    # img array -> resultados
    # ... omitimos detalle por dependencia opcional
    return {"vehicle_count": 0, "detection_confidence": 1.0, "congestion_level": 0}


## 13. Visualizaciones explicativas

Histograma de counts mock.


In [5]:
df_results = pd.DataFrame(results)
df_results["vehicle_count"].plot.hist(bins=10, color="#FF5722")
plt.title("Distribución de count mock"); plt.tight_layout()


## 14. Validaciones

Output válido y dentro de rango.


In [6]:
for r in results:
    assert 0 <= r["vehicle_count"] <= 200
    assert 0 <= r["detection_confidence"] <= 1
    assert r["congestion_level"] in (0, 1, 2, 3)
print("Mock OK")


Mock OK


## 15. Errores comunes

1. Devolver int vs float (TSDB espera float).
2. Re-leer la imagen en cada inferencia (cachear).
3. No filtrar `confidence < threshold`.


## 16. Ejercicios propuestos

1. Sustituye el mock por `ultralytics`.
2. Calcula la edad media del conjunto.
3. Mide latencia de la inferencia.


## 17. Cómo se reutiliza con datos reales

Cambiar `count_vehicles_mock` por `count_vehicles_real`.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `10_case_J_traffic_yolo/03_series_temporales_trafico.ipynb`.
- Documento web del caso: `docs/use-cases/case-j-traffic-yolo.md`.


## 19. Marco teórico (nivel doctoral)

### YOLO v8 — single-stage anchor-free detector

Por cada celda de la grid, salida:

$$
\hat{y} = (b_x, b_y, b_w, b_h, p_{obj}, p_{c_1}, ..., p_{c_C})
$$

Loss combinada:

$$
\mathcal{L} = \lambda_{box} \mathcal{L}_{CIoU} + \lambda_{obj} \mathcal{L}_{BCE,obj} + \lambda_{cls} \mathcal{L}_{BCE,cls}
$$

### Series temporales tráfico

$$
N_v(t) = \sum_{i=1}^{D_t} \mathbb{1}[\text{detection}_i \in v_{ROI}]
$$

con NMS IoU threshold = 0.5.

### Predictor congestión

$$
\hat{C}(t+15) = \text{XGB}(N_v(t), N_v(t-15), ..., \text{weather}, t_{hora}, t_{dow})
$$

con $C \in \{0, 1, 2, 3\}$ niveles de congestión.

### Métricas

$$
\text{mAP}@0.5 = \frac{1}{|C|} \sum_{c \in C} \text{AP}_c \quad (\text{IoU} \geq 0.5)
$$

Objetivos: mAP@0.5 ≥ 0.90 (car/truck), ≥ 0.75 (motorbike/bicycle).


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Aunque tangencial al BMS de aulas, este caso demuestra que la **stack de IA + datos sintéticos + modelos** de CAPTIA es extensible a otros verticales (smart cities). Activo comercial para diversificar.

### ROI estimado

| Concepto | Valor |
|---|---|
| Predicción congestión 15 min (semáforos) | +5 000 €/año |
| Detección incidentes < 60 s (emergencias) | +12 000 €/año |
| **Bruto** | **+17 000 €/año** |
| Compute GPU dedicada | -1 500 €/año |
| **Neto** | **+15 500 €/año** |


## 21. Bibliografía y referencias

- Redmon, J. & Farhadi, A. (2018). *YOLOv3: An Incremental Improvement*. arXiv:1804.02767.
- Ultralytics (2024). *YOLOv8 Documentation*. https://docs.ultralytics.com
- Lin, T.-Y. et al. (2014). *Microsoft COCO: Common Objects in Context*. ECCV.
- DGT España. *Información en tiempo real*. http://infocar.dgt.es
